In [0]:
dbutils.widgets.text("CustomerCode_Source", "BOSHOSP")

dbutils.widgets.text(
    "run_id",
    "472519629468310"
)
run_id=dbutils.widgets.get("run_id")

In [0]:
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from datetime import datetime

# 1. Configuration & Credentials
SMTP_SERVER = "smtp.gmail.com"
SMTP_PORT = 587
SENDER_EMAIL = "naveenreddymg408@gmail.com"
CUSTOMER_EMAIL = "naveenamg568@gmail.com"

# ============================================================
# PASSWORD SETUP OPTIONS:
# ============================================================

# OPTION 1: Use Secret Scope (Recommended - uncomment after creating scope)
# To create: databricks secrets create-scope gmail_api_key
#            databricks secrets put-secret gmail_api_key secret_value
# SENDER_PASSWORD = dbutils.secrets.get(scope="gmail_api_key", key="secret_value")

# OPTION 2: Hardcode for testing (Replace with your actual Gmail App Password)
SENDER_PASSWORD = "jqct igxd waox orrh"  # ⚠️ Replace this!

# ============================================================
# NOTE: To get a Gmail App Password:
# 1. Go to https://myaccount.google.com/security
# 2. Enable 2-Step Verification if not already enabled
# 3. Go to "App passwords" and generate a new one for "Mail"
# 4. Copy that 16-character password (no spaces)
# 5. Use it when running: databricks secrets put-secret gmail_api_key secret_value
# ============================================================

# 2. Execute Spark SQL Query
run_id_int = int(run_id)
run_details_df = spark.sql(f"""SELECT * FROM clinicalforge.metadata.pipelinerun WHERE RunID={run_id_int}""")
display(run_details_df)

# Convert to Pandas for email formatting
pandas_df = run_details_df.toPandas()

# 3. Extract pipeline run details
if not pandas_df.empty:
    # Extract summary info from first row for subject line
    first_row = pandas_df.iloc[0]
    customer_id = str(first_row['HealthClientID'])
    overall_status = "SUCCESS" if all(pandas_df['RunStatus'] == 'SUCCESS') else "FAILED"
    
    # Calculate totals across all records
    total_records = int(pandas_df['RecordsIngested'].sum())
    total_tables = len(pandas_df)
    
    # Build HTML table rows for all records
    table_rows = ""
    for idx, row in pandas_df.iterrows():
        status_class = "status-success" if row['RunStatus'] == 'SUCCESS' else "status-failed"
        
        # Calculate duration for each row
        if row['StartDateTime'] and row['EndDateTime']:
            duration = row['EndDateTime'] - row['StartDateTime']
            duration_str = str(duration)
        else:
            duration_str = "N/A"
        
        error_display = str(row['ErrorMessage']) if row['ErrorMessage'] else "None"
        
        table_rows += f"""
        <tr>
            <td>{row['TableID']}</td>
            <td>{row['TargetTableName']}</td>
            <td class="{status_class}">{row['RunStatus']}</td>
            <td>{row['RecordsIngested']:,}</td>
            <td>{row['StartDateTime']}</td>
            <td>{row['EndDateTime']}</td>
            <td>{duration_str}</td>
            <td>{error_display}</td>
        </tr>
        """
else:
    customer_id = "Unknown"
    overall_status = "No Data"
    total_records = 0
    total_tables = 0
    table_rows = "<tr><td colspan='8'>No data found</td></tr>"

# 4. Construct the Email Body (HTML) - Table format for all records
styled_email_body = f"""
<html>
<head>
<style>
  body {{ font-family: Arial, sans-serif; color: #333; line-height: 1.6; }}
  .header {{ background-color: #4CAF50; color: white; padding: 20px; text-align: center; }}
  .content {{ padding: 20px; }}
  .section {{ margin-bottom: 20px; }}
  .label {{ font-weight: bold; color: #555; }}
  .value {{ color: #000; margin-left: 10px; }}
  .status-success {{ color: #4CAF50; font-weight: bold; }}
  .status-failed {{ color: #f44336; font-weight: bold; }}
  .footer {{ background-color: #f5f5f5; padding: 15px; text-align: center; margin-top: 30px; }}
  table {{ border-collapse: collapse; width: 100%; margin-top: 15px; }}
  th {{ background-color: #4CAF50; color: white; padding: 12px; text-align: left; }}
  td {{ border: 1px solid #ddd; padding: 10px; }}
  tr:nth-child(even) {{ background-color: #f9f9f9; }}
  tr:hover {{ background-color: #f1f1f1; }}
</style>
</head>
<body>
  <div class="header">
    <h2>Pipeline Execution Report</h2>
  </div>
  
  <div class="content">
    <div class="section">
      <h3>Execution Summary</h3>
      <p><span class="label">Run ID:</span><span class="value">{run_id}</span></p>
      <p><span class="label">Customer/Client ID:</span><span class="value">{customer_id}</span></p>
      <p><span class="label">Overall Status:</span><span class="value {'status-success' if overall_status == 'SUCCESS' else 'status-failed'}">{overall_status}</span></p>
      <p><span class="label">Total Tables Processed:</span><span class="value">{total_tables}</span></p>
      <p><span class="label">Total Records Ingested:</span><span class="value">{total_records:,}</span></p>
    </div>
    
    <div class="section">
      <h3>Detailed Pipeline Results</h3>
      <table>
        <thead>
          <tr>
            <th>Table ID</th>
            <th>Target Table</th>
            <th>Status</th>
            <th>Records</th>
            <th>Start Time</th>
            <th>End Time</th>
            <th>Duration</th>
            <th>Error Message</th>
          </tr>
        </thead>
        <tbody>
          {table_rows}
        </tbody>
      </table>
    </div>
  </div>
  
  <div class="footer">
    <p>This is an automated notification from ClinicalForge Pipeline Monitoring System</p>
    <p><strong>ClinicalForge Support Team</strong></p>
    <p style="font-size: 12px; color: #888;">For questions or support, please contact your system administrator</p>
  </div>
</body>
</html>
"""

# 5. Create the Email Message with Dynamic Subject
message = MIMEMultipart()
message["From"] = SENDER_EMAIL
message["To"] = CUSTOMER_EMAIL
message["Subject"] = f"Pipeline {overall_status} - {customer_id} - {total_tables} Tables - Run #{run_id}"

# Attach HTML Content
message.attach(MIMEText(styled_email_body, "html"))

# 6. Connect to Gmail and Send
try:
    server = smtplib.SMTP(SMTP_SERVER, SMTP_PORT)
    server.starttls()
    server.login(SENDER_EMAIL, SENDER_PASSWORD)
    server.sendmail(SENDER_EMAIL, CUSTOMER_EMAIL, message.as_string())
    print(f"✅ Email successfully sent to {CUSTOMER_EMAIL}")
    print(f"📧 Subject: {message['Subject']}")
    print(f"📊 Overall Status: {overall_status}")
    print(f"📋 Tables Processed: {total_tables}")
    print(f"📈 Total Records: {total_records:,}")
except Exception as e:
    print(f"❌ Failed to send email: {e}")
finally:
    if 'server' in locals():
        server.quit()
